# GTM Deep Agent — LinkedIn & Email content for The Gen Academy

A hands-on tutorial of the **Deep Agents** framework — LangChain's batteries-included
agent layer that sits on top of LangGraph.

We build a **GTM content assistant**: an orchestrator that delegates to specialist
subagents to draft a LinkedIn post and a marketing email, grounded in The Gen Academy's
real brand.

> **Draft-only.** Nothing is ever actually posted or emailed — the "publish" step is a
> simulation behind a human-approval gate.

## Architecture at a glance

```
                    ┌─────────────────────────────────┐
   Campaign brief   │      GTM Orchestrator Agent      │
        │           │  (plans, delegates, assembles)   │
        └──────────▶│  write_todos → task → publish    │
                    └───┬─────────┬─────────┬──────┬────┘
                        │ task    │ task    │ task │ task
                   ┌────▼───┐ ┌───▼────┐ ┌──▼───┐ ┌▼──────┐
                   │research│ │linkedin│ │email │ │critic │
                   │        │ │ writer │ │writer│ │       │
                   │ web    │ │ (LI    │ │(email│ │(linters│
                   │ search │ │ skill) │ │skill)│ │ both) │
                   └────┬───┘ └───┬────┘ └──┬───┘ └──┬────┘
                        └─────────┴────┬────┴────────┘
                              ┌────────▼─────────┐
                              │   SHARED STATE   │
                              │ files + todos    │  ← drafts pass between agents here
                              └──────────────────┘
   Persistence:  disk (/research, /drafts, /final)   +   Store (/memories — cross-session)
```

**Deep Agents concepts this notebook demonstrates:**

| Concept | Where you'll see it |
|---|---|
| **Planning** | `write_todos` breaks the brief into steps |
| **Subagents** | `task` delegates to specialists, each with its own tools, prompt, and *isolated context* |
| **Shared state** | message histories are isolated, but the **filesystem + todos are shared** — drafts pass between agents via files |
| **Skills** | `SKILL.md` style guides loaded on demand (progressive disclosure) |
| **Memory** | `/memories/` persists across sessions via a `Store` |
| **Human-in-the-loop** | the `publish` action pauses for your approval |
| **Runaway protection** | `recursion_limit` is the hard backstop on the agent loop |

![Diagram](./images/01_img.png)

## 1. Install & API keys

Dependencies are managed by **uv** (see `pyproject.toml`). If you opened this notebook with the project's kernel, everything is already installed.

In [1]:
# Already handled by `uv sync`. Uncomment to install into the current kernel if needed:
# %pip install -q deepagents "langchain>=1.0,<2.0" langchain-openai langchain-tavily python-dotenv

In [1]:
import os, getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads .env if present

def _need(key, prompt):
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(prompt)

_need("OPENAI_API_KEY", "OpenAI API key: ")
# Tavily is OPTIONAL — press Enter to skip and run grounding-only.
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key (Enter to skip): ")

# LangSmith tracing is OPTIONAL — set LANGSMITH_API_KEY in .env to turn it on.
# When enabled, every run uploads a nested trace tree to smith.langchain.com:
# the orchestrator at the top, each `task` call as a child span, and the full
# subagent trajectory (LLM calls + tool calls) nested inside it — i.e. exactly
# "which agent ran which tool", which result["messages"] can't show you.
if os.environ.get("LANGSMITH_API_KEY", "").strip():
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "gtm-deep-agent")
    os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
    print(f"LangSmith tracing: ENABLED → project '{os.environ['LANGSMITH_PROJECT']}'")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: disabled (no LANGSMITH_API_KEY in .env)")

MODEL = "gpt-5.5"
print("Model:", MODEL)

LangSmith tracing: ENABLED → project 'gtm-deep-agent'
Model: gpt-5.5


## 2. Company grounding

We ground the agent in real facts about The Gen Academy (scraped from the website into a Markdown file). Grounding is how we keep an LLM from inventing claims — the agent is told to honor this and not go beyond it.

In [2]:
GROUNDING = Path("grounding/gen_academy.md").read_text()
print(GROUNDING)

# Grounding: The Gen Academy

> Source: thegenacademy.com (fetched 2026-06-13). Brand context for the GTM content agent.

## Mission
AI is the most transformative technology of our generation, but it shouldn't be
accessible only to those at top universities or big tech companies. The Gen Academy
democratizes world-class AI education through **practical, accessible, community-driven**
learning.

## Audience (ICP)
Professionals advancing their careers in AI — from the AI-curious through to experienced
engineers wanting to architect production-scale systems.
- 50,000+ learners reached
- 2,500+ professionals trained and upskilled into high-impact roles

## Flagship Offering
**Mastering Agentic AI Certification**
- 6-week intensive + bonus 7th week on career development
- Battle-tested curriculum: theory blended with hands-on projects and a capstone
- Guest instructors from Google, Microsoft, Adobe, Fireworks AI
- Career coaching: job-market navigation + portfolio building

## Value Proposi

## 3. Tools

Three kinds of tools, each teaching a distinct idea:

- **`web_search` (Tavily)** — the agent reaching *outside itself* for fresh info. Optional.
- **`linkedin_post_linter` / `email_linter`** — deterministic validators. Teaching point:
  the LLM **drafts**, but plain code **validates** what models are unreliable at
  (counting characters, enforcing hard limits).
- **`publish_post`** — a **simulated** outbound action, wrapped later in a human-approval gate.

![Diagram](./images/02_img.png)

In [3]:
from langchain.tools import tool

# --- Tool 1: Web search (external API) — optional ---------------------
research_tools = []
if os.environ.get("TAVILY_API_KEY", "").strip():
    from langchain_tavily import TavilySearch
    research_tools = [TavilySearch(max_results=3)]
    print("Web search: Tavily ENABLED")
else:
    print("Web search: disabled (no key) — researcher will rely on grounding")

# --- Tools 2 & 3: Deterministic linters -------------------------------
@tool
def linkedin_post_linter(text: str) -> str:
    """Validate a LinkedIn post (length, hook, hashtags, CTA). Returns a pass/fail report."""
    issues = []
    if len(text) > 3000:
        issues.append(f"Too long: {len(text)}/3000 chars")
    hook = (text.strip().splitlines() or [""])[0]
    if len(hook) > 250:
        issues.append(f"Hook is {len(hook)} chars; keep < 250 so it shows before 'see more'")
    hashtags = [w for w in text.split() if w.startswith("#")]
    if len(hashtags) > 5:
        issues.append(f"Too many hashtags: {len(hashtags)} (use 3-5)")
    cta_words = ["http", "comment", "dm", "join", "link", "register", "apply", "sign up"]
    if not any(w in text.lower() for w in cta_words):
        issues.append("No clear call-to-action detected")
    return "PASS" if not issues else "ISSUES:\n- " + "\n- ".join(issues)

@tool
def email_linter(subject: str, body: str) -> str:
    """Validate a marketing email (subject length, single CTA, spam words). Returns a report."""
    issues = []
    if not subject.strip():
        issues.append("Missing subject line")
    elif len(subject) > 60:
        issues.append(f"Subject too long: {len(subject)}/60 chars (mobile clips ~50)")
    spam = [w for w in ["free", "guarantee", "act now", "limited time", "!!!", "100%"]
            if w in (subject + " " + body).lower()]
    if spam:
        issues.append(f"Spam-trigger words: {', '.join(spam)}")
    ctas = sum(body.lower().count(k) for k in ["sign up", "register", "apply", "book", "join"])
    if ctas == 0:
        issues.append("No call-to-action found")
    elif ctas > 2:
        issues.append(f"Multiple CTAs ({ctas}); one clear CTA converts better")
    return "PASS" if not issues else "ISSUES:\n- " + "\n- ".join(issues)

# --- Tool 4: Mock publish (SIMULATION ONLY) ---------------------------
@tool
def publish_post(channel: str, content: str) -> str:
    """Publish content to a channel (SIMULATED — nothing is actually sent). channel: 'linkedin' or 'email'."""
    return f"[SIMULATED] Published to {channel} ({len(content)} chars). Nothing was actually sent."

print("Tools ready.")

Web search: Tavily ENABLED
Tools ready.


## 4. Skills (progressive disclosure)

A **skill** is a `SKILL.md` file with YAML frontmatter (`name` + `description`). The agent
only sees the *description* until it decides a skill is relevant — then it loads the full
body. This keeps the system prompt lean. Our two skills are the channel style guides.

![Diagram](./images/03_img.png)

In [4]:
print(Path("skills/linkedin-style/SKILL.md").read_text())
print("\n" + "=" * 70 + "\n")
print(Path("skills/email-style/SKILL.md").read_text())

---
name: linkedin-style
description: How to write a LinkedIn post in The Gen Academy voice — hook-first, short, one CTA.
---

# LinkedIn Style

## When to use
When drafting a LinkedIn post for The Gen Academy.

## Structure
1. **Hook** (first line, < 250 chars): a bold claim, question, or surprising stat. It must stand alone — everything after is hidden behind "…see more".
2. **Body** (1–3 short paragraphs): one idea each. Use line breaks, not walls of text.
3. **Proof**: a concrete number or outcome (e.g. "2,500+ engineers upskilled").
4. **CTA** (exactly one): comment, DM, or a link to apply/join.

## Voice
Direct, energetic, confident. "Ship real work. Land real opportunities."
Speak to engineers as peers. No corporate filler.

## Rules
- Under 3,000 characters; aim for 600–1,200.
- 3–5 relevant hashtags max, at the end.
- One CTA only. Sparing emoji (0–3).
- No hype words ("revolutionary", "game-changing"). Show, don't boast.



---
name: email-style
description: How to write a ma

## 5. Backends: files + cross-session memory

Two persistence layers, wired with a **`CompositeBackend`** that routes by path:

- **`/memories/*`** → `StoreBackend` — survives across sessions (brand voice we learn once)
- **everything else** → `FilesystemBackend` on real disk — so `/drafts` and `/final` show
  up in your file explorer during the demo

The **checkpointer** holds per-thread state and is *required* for the human-in-the-loop gate.

![Diagram](./images/04_img.png)

In [5]:
from deepagents.backends import CompositeBackend, FilesystemBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

store = InMemoryStore()        # cross-session memory (use a PostgresStore in production)
checkpointer = MemorySaver()   # per-thread state + required for HITL interrupts

# resolves the store/runtime itself via get_store(); the `store=` we hand to
# create_deep_agent below is what it picks up. `namespace` sets where memories
backend = CompositeBackend(
    default=FilesystemBackend(root_dir=".", virtual_mode=True),
    routes={"/memories/": StoreBackend(namespace=lambda ctx: ("filesystem",))},
)
print("Backend: disk (default) + Store (/memories/) | checkpointer + store ready")

Backend: disk (default) + Store (/memories/) | checkpointer + store ready


## 6. Specialist subagents

Each subagent is a dict: a name, a description (the orchestrator reads this to decide *when*
to delegate), a focused system prompt, its own tools, and its own skills. Subagents are
**stateless** — each `task` call is a fresh run — so instructions must be complete.

Note the **division of labour**: each writer only gets *its* linter and the style skills;
the researcher only gets web search. Clean tools = clean context.

![Diagram](./images/05_img.png)

In [6]:
SKILLS = ["/skills/"]  # loaded from disk via the FilesystemBackend

research_subagent = {
    "name": "researcher",
    "description": "Gathers external context on the campaign topic and writes a short brief. Use first.",
    "system_prompt": (
        "You research a topic for a marketing campaign. Use web_search if available; "
        "otherwise summarise from what the orchestrator gives you. "
        "Write 5-8 crisp bullet points to /research/brief.md and return a one-line summary."
    ),
    "tools": research_tools,
}

linkedin_subagent = {
    "name": "linkedin-writer",
    "description": "Drafts ONE LinkedIn post in The Gen Academy voice. Use after research.",
    "system_prompt": (
        "Load the 'linkedin-style' skill and follow it strictly. Read /research/brief.md if it exists. "
        "Draft one LinkedIn post, run linkedin_post_linter, fix any issues it reports, "
        "then save the final post to /drafts/linkedin.md. Return the final post text."
    ),
    "tools": [linkedin_post_linter],
    "skills": SKILLS,
}

email_subagent = {
    "name": "email-writer",
    "description": "Drafts ONE marketing email in The Gen Academy voice. Use after research.",
    "system_prompt": (
        "Load the 'email-style' skill and follow it strictly. Read /research/brief.md if it exists. "
        "Draft one email (subject + body), run email_linter, fix any issues it reports, "
        "then save it to /drafts/email.md. Return the subject and body."
    ),
    "tools": [email_linter],
    "skills": SKILLS,
}

critic_subagent = {
    "name": "critic",
    "description": (
        "Reviews both drafts for brand voice, skill compliance, factual accuracy, and quality; "
        "runs the linters; suggests fixes."
    ),
    "system_prompt": (
        "Read /drafts/linkedin.md and /drafts/email.md. Review each draft on four axes:\n"
        "1. BRAND VOICE — judge against the brand voice (direct, energetic, outcome-focused; "
        "'Ship real work. Land real opportunities.').\n"
        "2. SKILL COMPLIANCE — load the 'linkedin-style' and 'email-style' skills and check each "
        "draft actually follows its channel's skill rules (structure, format, do's and don'ts).\n"
        "3. FACTUAL ACCURACY — read /research/brief.md and verify every claim, statistic, and detail "
        "in the drafts is supported by the brief (or the brand grounding). Flag anything invented, "
        "unsupported, or contradicting the research.\n"
        "4. LINTERS — run linkedin_post_linter and email_linter.\n"
        "Return a short review with concrete, actionable fixes, calling out any skill violations and "
        "unsupported facts explicitly."
    ),
    "tools": [linkedin_post_linter, email_linter],
    "skills": SKILLS,
}

print("4 subagents defined.")

4 subagents defined.


## 7. Assemble the deep agent

This is the whole point of Deep Agents: you **configure**, you don't build the plumbing. Planning, the filesystem tools, and delegation come for free.

![Diagram](./images/06_img.png)

In [7]:
from deepagents import create_deep_agent

ORCHESTRATOR_PROMPT = f"""You are a GTM content lead for The Gen Academy.
For a campaign brief you produce one LinkedIn post and one marketing email.

Brand grounding (always honor this; never invent facts beyond it + research):
{GROUNDING}

Workflow — first call write_todos to plan, then:
1. Delegate to 'researcher' to gather context (writes /research/brief.md).
2. Delegate to 'linkedin-writer' (writes /drafts/linkedin.md).
3. Delegate to 'email-writer' (writes /drafts/email.md).
4. Delegate to 'critic' to review both; if it flags something important, re-delegate to fix it.
5. Save the approved versions to /final/linkedin.md and /final/email.md.
6. Save a one-paragraph brand-voice note to /memories/brand_voice.md for future sessions.
7. Finally, publish BOTH channels. In a SINGLE step, emit two publish_post tool calls
   together — one with channel='linkedin' and one with channel='email'. Do not write any
   summary or final message until BOTH publish_post calls have been made and returned.

Keep everything concise and on-brand."""

agent = create_deep_agent(
    model=MODEL,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[publish_post],
    subagents=[research_subagent, linkedin_subagent, email_subagent, critic_subagent],
    backend=backend,
    store=store,
    skills=SKILLS,
    interrupt_on={"publish_post": True},   # HITL gate — needs the checkpointer
    checkpointer=checkpointer,
)
print("Deep agent assembled.")

Deep agent assembled.


## 8. Run a campaign brief

We pass one brief and let the orchestrator plan and delegate. Watch the message stream:
you'll see the orchestrator's `task` calls and each subagent's **final report** — but *not*
the subagents' internal back-and-forth. That hidden churn is the context-isolation benefit.

`recursion_limit` caps the agent loop — the hard backstop against runaway loops.

![Diagram](./images/07_img.png)

In [8]:
config = {"configurable": {"thread_id": "gtm-demo-1"}, "recursion_limit": 60}

brief = (
    "Campaign: enrollment push for the next cohort of the Mastering Agentic AI "
    "Certification. Audience: working software engineers who are AI-curious. "
    "Angle: go from AI-curious to production-ready in 6 weeks."
)

result = agent.invoke({"messages": [{"role": "user", "content": brief}]}, config=config) 
#try streaming the output to see how the agent is executing

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Campaign: enrollment push for the next cohort of the Mastering Agentic AI Certification. Audience: working software engineers who are AI-curious. Angle: go from AI-curious to production-ready in 6 weeks.
================================== Ai Message ==================================
Tool Calls:
  write_todos (call_bMRcjTP2L5U4m7qTLlurtuPk)
 Call ID: call_bMRcjTP2L5U4m7qTLlurtuPk
  Args:
    todos: [{'content': 'Gather campaign context and write /research/brief.md', 'status': 'in_progress'}, {'content': 'Draft LinkedIn post via linkedin-writer', 'status': 'pending'}, {'content': 'Draft marketing email via email-writer', 'status': 'pending'}, {'content': 'Critic review and fix if needed', 'status': 'pending'}, {'content': 'Save approved final files and brand voice memory', 'status': 'pending'}, {'content': 'Publish LinkedIn and email', 'status': 'pending'}]
================================= Tool Message ==

In [9]:
result

{'messages': [HumanMessage(content='Campaign: enrollment push for the next cohort of the Mastering Agentic AI Certification. Audience: working software engineers who are AI-curious. Angle: go from AI-curious to production-ready in 6 weeks.', additional_kwargs={}, response_metadata={}, id='127aff97-0286-4270-bd5e-fc17e7d4f4ac'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 6766, 'total_tokens': 6852, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 4736}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-EGkV0aET4PxNAOVA9WcUBUrCOPpzt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a038e1-814c-7a13-945a-135f7cb151f5-0', tool_calls=[{'name': 'wri

![Diagram](./images/08_img.png)

In [10]:
# Inspect the SHARED STATE the agents produced (drafts pass between them via these files).
for path in ["research/brief.md", "drafts/linkedin.md", "drafts/email.md",
             "final/linkedin.md", "final/email.md"]:
    p = Path(path)
    if p.exists():
        print(f"\n===== /{path} =====\n{p.read_text()}")

# The plan the orchestrator tracked:
print("\n===== TODOS =====")
for t in result.get("todos", []):
    print(f"[{t.get('status')}] {t.get('content')}")


===== /research/brief.md =====
# The Gen Academy: Mastering Agentic AI Certification — Enrollment Campaign Brief

- **Audience insight:** Working software engineers are AI-curious but skeptical: Stack Overflow’s 2025 Developer Survey says more developers distrust AI-tool accuracy (46%) than trust it (33%), with experienced developers especially cautious. Message to engineers who want the judgment to use AI responsibly, not just prompts. Source: https://survey.stackoverflow.co/2025/ai
- **Market context:** AI adoption is mainstream, but agentic AI is still early in enterprise: McKinsey’s 2025 State of AI reports nearly 9 in 10 organizations regularly use AI, while 23% are scaling an agentic AI system and 39% are experimenting. The opportunity: become the engineer who can move teams from experiments to shipped workflows. Source: https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai
- **Skills demand:** Stanford HAI’s AI Index notes AI job demand is shifting fro

## 9. Human-in-the-loop: approve the publish gate

We set `interrupt_on={"publish_post": True}`, so the run **pauses** before each publish and
hands control back to us. We inspect what it wants to do, then approve (or reject with
feedback, or edit the arguments) and resume with a `Command`.

![Diagram](./images/09_img.png)

In [9]:
from langgraph.types import Command

# What is the agent waiting to do? In this LangChain version each interrupt's
# .value is a SINGLE HITLRequest dict: {"action_requests": [...], "review_configs": [...]}.
# One action_request == one hanging tool call, and we must send one decision PER
# action_request — not one per interrupt.
#
# How MANY action_requests appear depends on what the model emitted: if it issues
# both publish_post calls in one step (as the orchestrator prompt now instructs),
# they batch into a single request with TWO action_requests. If it only emits one,
# you'll see one here, approve it, resume — and the second call surfaces as the next
# interrupt on the following loop iteration. The while-loop below handles either case.
def pending_requests(res):
    reqs = []
    for intr in res.get("__interrupt__", []):
        val = intr.value
        if isinstance(val, dict) and "action_requests" in val:
            reqs.extend(val["action_requests"])      # unwrap the HITLRequest
        elif isinstance(val, list):
            reqs.extend(val)                          # older list-style payload
        else:
            reqs.append(val)
    return reqs

reqs = pending_requests(result)
if reqs:
    for r in reqs:
        print("PENDING APPROVAL:", r)
else:
    print("No interrupt pending (agent may have finished without calling publish).")

# Approve each pending publish and resume. One decision PER hanging tool call.
# Loops until no interrupt remains — so it catches publish calls that arrive across
# multiple turns, not just a single batched request.
# To reject instead, use: {"type": "reject", "message": "make it punchier"}
while "__interrupt__" in result:
    n = len(pending_requests(result))
    decisions = [{"type": "approve"} for _ in range(n)]
    result = agent.invoke(Command(resume={"decisions": decisions}), config=config)

# Sanity check: confirm BOTH channels were actually published.
published = [
    tc["args"].get("channel")
    for m in result["messages"]
    for tc in (getattr(m, "tool_calls", None) or [])
    if tc["name"] == "publish_post"
]
print("\nPublished channels:", published)
for ch in ("linkedin", "email"):
    if ch not in published:
        print(f"  WARNING: {ch} was never published — the model skipped its publish_post call.")

print("\n===== FINAL RESPONSE =====")
result["messages"][-1].pretty_print()

PENDING APPROVAL: {'name': 'publish_post', 'args': {'channel': 'linkedin', 'content': 'AI-curious engineers don’t need another prompt list. They need reps shipping real systems.\n\nIf you’ve tried AI tools, you already know the gap: using AI is not the same as building AI you can explain, evaluate, and trust in production.\n\nMastering Agentic AI Certification is a 6-week intensive built for working software engineers who want production-ready knowledge, not theory. You’ll build hands-on projects, complete a capstone, and learn from guest instructors from Google, Microsoft, Adobe, and Fireworks AI.\n\nThen there’s a bonus 7th career week focused on portfolio building, job-market navigation, and career coaching.\n\nThe Gen Academy has reached 50,000+ learners and trained/upskilled 2,500+ professionals, with curriculum distilled from 7 years building AI systems at major tech companies.\n\nShip real work. Land real opportunities.\n\nApply for the next cohort of Mastering Agentic AI Certif

## 10. Persistent memory across sessions

A **new thread** = a new session with a fresh message history. But anything written to
`/memories/` lives in the `Store`, not the thread — so a brand-new conversation can still
recall the brand voice we saved earlier. This is the difference between *thread state* and
*long-term memory*.

![Diagram](./images/10_img.png)

In [10]:
config2 = {"configurable": {"thread_id": "gtm-demo-2"}, "recursion_limit": 20}
result2 = agent.invoke(
    {"messages": [{"role": "user",
                   "content": "Read /memories/brand_voice.md and summarise our brand voice in one line."}]},
    config=config2,
)
result2["messages"][-1].pretty_print()

================================== Ai Message ==================================

Direct, energetic, and outcome-focused: practical AI education without hype, grounded in honest technical sharpness, hands-on proof, and career outcomes.


## Recap

You built a multi-agent GTM content system with almost no plumbing code:

- An **orchestrator** that *planned* (`write_todos`) and *delegated* (`task`).
- Four **subagents** with isolated context and focused tools.
- **Skills** for channel style, loaded on demand.
- A **composite backend**: disk for artifacts, a Store for cross-session memory.
- A **human-in-the-loop** gate on the (simulated) publish step.
- `recursion_limit` as the safety backstop.

**Where to take it next:** swap `InMemoryStore` for a `PostgresStore`, add a real Gmail
draft tool behind the HITL gate, or give the critic a structured `response_format` so its
verdict is machine-readable.